In [1]:
import pandas as pd
import requests
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_POPULAR
from transformers import pipeline

In [1]:
#downloader = YoutubeCommentDownloader()
#comments = downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR)

In [5]:
urls = [
    'https://www.youtube.com/watch?v=3g2YoTBrQwI',
    'https://www.youtube.com/watch?v=cNS72JIULeY',
    'https://www.youtube.com/watch?v=yULYMOt1xlQ',
    'https://www.youtube.com/watch?v=zrnBhyeEYjw',
    'https://www.youtube.com/watch?v=LXpmlcTpZsU'
]

video_names = [
    'Fondasi Keropos Pendidikan Indonesia',
    'Salah Kaprah dalam Belajar',
    'Sabda PS - IQ, Kecerdasan, dan Pendidikan',
    'Faktor Penting yang Menentukan Kualitas Pendidikan',
    'Paradigma Masyarakat Baru'
]

downloader = YoutubeCommentDownloader()

comment_list = []

for url, video_name in zip(urls, video_names):

    print(f"Sedang mengambil komentar: {video_name}")

    count = 0

    comments = downloader.get_comments_from_url(
        url,
        sort_by=SORT_BY_POPULAR
    )

    for comment in comments:

        comment_list.append({
            'Video': video_name,
            'URL': url,
            'User': comment['author'],
            'Comment': comment['text']
        })

        count += 1

        if count >= 1000:
            break

    print(f"Berhasil mengambil {count} komentar\n")


# Buat DataFrame
df = pd.DataFrame(comment_list)

# Hapus komentar duplikat
df = df.drop_duplicates(subset=['Comment'])

# Reset index
df = df.reset_index(drop=True)

print(df.head())
print("\nJumlah data:", len(df))

Sedang mengambil komentar: Fondasi Keropos Pendidikan Indonesia
Berhasil mengambil 1000 komentar

Sedang mengambil komentar: Salah Kaprah dalam Belajar
Berhasil mengambil 626 komentar

Sedang mengambil komentar: Sabda PS - IQ, Kecerdasan, dan Pendidikan
Berhasil mengambil 511 komentar

Sedang mengambil komentar: Faktor Penting yang Menentukan Kualitas Pendidikan
Berhasil mengambil 161 komentar

Sedang mengambil komentar: Paradigma Masyarakat Baru
Berhasil mengambil 711 komentar

                                  Video  \
0  Fondasi Keropos Pendidikan Indonesia   
1  Fondasi Keropos Pendidikan Indonesia   
2  Fondasi Keropos Pendidikan Indonesia   
3  Fondasi Keropos Pendidikan Indonesia   
4  Fondasi Keropos Pendidikan Indonesia   

                                           URL                       User  \
0  https://www.youtube.com/watch?v=3g2YoTBrQwI           @MalakaProjectid   
1  https://www.youtube.com/watch?v=3g2YoTBrQwI  @maulititusekopramono9909   
2  https://www.youtube.com

In [6]:
# Jumlah data
print("Jumlah data:", len(df))

# Cek missing value
print("\nMissing value:")
print(df.isnull().sum())

# Cek duplikat
print("\nJumlah duplikat:")
print(df.duplicated(subset=['Video', 'User', 'Comment']).sum())

# Jumlah komentar per video
print("\nKomentar per video:")
print(df['Video'].value_counts())

# Panjang komentar
df['Panjang_Komentar'] = df['Comment'].str.len()

print("\nStatistik panjang komentar:")
print(df['Panjang_Komentar'].describe())

Jumlah data: 2979

Missing value:
Video      0
URL        0
User       0
Comment    0
dtype: int64

Jumlah duplikat:
0

Komentar per video:
Video
Fondasi Keropos Pendidikan Indonesia                  994
Paradigma Masyarakat Baru                             689
Salah Kaprah dalam Belajar                            624
Sabda PS - IQ, Kecerdasan, dan Pendidikan             511
Faktor Penting yang Menentukan Kualitas Pendidikan    161
Name: count, dtype: int64

Statistik panjang komentar:
count    2979.000000
mean      229.765022
std       320.383286
min         1.000000
25%        53.000000
50%       122.000000
75%       282.500000
max      6120.000000
Name: Panjang_Komentar, dtype: float64


In [7]:
import re

def clean_text(text):
    text = text.lower()
    
    # Hapus URL
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Hapus mention
    text = re.sub(r'@\w+', '', text)
    
    # Hapus hashtag tapi tetap mempertahankan katanya
    text = re.sub(r'#', '', text)
    
    # Hapus karakter selain huruf dan angka
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    # Hapus angka
    text = re.sub(r'\d+', ' ', text)
    
    # Hapus spasi berlebihan
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_text'] = df['Comment'].apply(clean_text)

df[['Comment', 'clean_text']].head(10)

,Comment,clean_text
0,Tonton videonya sampai habis untuk tahu kondis...,tonton videonya sampai habis untuk tahu kondis...
1,Menurut pengalaman saya yang pernah mengajar d...,menurut pengalaman saya yang pernah mengajar d...
2,"Dulu ada istilah, Indonesia tidak kkurangan or...",dulu ada istilah indonesia tidak kkurangan orn...
3,Jadi keinget ada teori \nbahwa sengaja pendidi...,jadi keinget ada teori bahwa sengaja pendidika...
4,Wah keren sekali ka idenya. Sebagai guru saya ...,wah keren sekali ka idenya sebagai guru saya s...
5,"Setelah scroll komentar, sangat mencerminkan ...",setelah scroll komentar sangat mencerminkan ap...
6,"Selain banyak faktor, faktor utama yaitu guru ...",selain banyak faktor faktor utama yaitu guru t...
7,"3:39 Pernah ikut seminari oleh PPI Oslo, Finla...",pernah ikut seminari oleh ppi oslo finlandia i...
8,"Sbg seorang guru, udah mati2an ngasih materi j...",sbg seorang guru udah mati an ngasih materi ju...
9,"Pejabat : ""kami tidak akan membiarkan rakyat p...",pejabat kami tidak akan membiarkan rakyat pint...


In [8]:
!pip install transformers torch sentencepiece

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ------------------------------ --------- 1.0/1.4 MB 3.0 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.6 MB/s  0:00:00


In [9]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

C:\Users\LENOVO\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--w11wo--indonesian-roberta-base-sentiment-classifier. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\LENOVO\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarni

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [10]:
sample = df['clean_text'].head(10).tolist()

hasil = sentiment_pipeline(sample)

for text, result in zip(sample, hasil):
    print("Komentar :", text)
    print("Sentimen :", result['label'])
    print("Score    :", round(result['score'], 4))
    print("-" * 60)

Komentar : tonton videonya sampai habis untuk tahu kondisi sebenarnya dan langkah realistis yang bisa kita ambil untuk memperbaiki pendidikan dasar di indonesia
Sentimen : positive
Score    : 0.6666
------------------------------------------------------------
Komentar : menurut pengalaman saya yang pernah mengajar di sekolah swasta bonafit dan sekolah negeri pinggiran salah satu pendidik yang tidak dipersiapkan dengan baik adalah orang tua
Sentimen : negative
Score    : 0.6188
------------------------------------------------------------
Komentar : dulu ada istilah indonesia tidak kkurangan orng pintar tp kekurangan orang jujur tapi sekarang indonesia kekurangan dua duanya
Sentimen : negative
Score    : 0.9559
------------------------------------------------------------
Komentar : jadi keinget ada teori bahwa sengaja pendidikan kita dibiarkan jelek oleh pemerintah agar bisa terus korupsi
Sentimen : negative
Score    : 0.9956
------------------------------------------------------------
K

In [11]:
df.to_csv(
    'dataset_cania_2979_clean.csv',
    index=False,
    encoding='utf-8-sig'
)

In [12]:
import os

os.makedirs(r'D:\Data Sains', exist_ok=True)

df.to_csv(
    r'D:\Data Sains\dataset_cania_2979_clean.csv',
    index=False,
    encoding='utf-8-sig'
)

print("File berhasil disimpan!")

File berhasil disimpan!


In [13]:
df.head(10)

,Video,URL,User,Comment,Panjang_Komentar,clean_text
0,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@MalakaProjectid,Tonton videonya sampai habis untuk tahu kondis...,150,tonton videonya sampai habis untuk tahu kondis...
1,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@maulititusekopramono9909,Menurut pengalaman saya yang pernah mengajar d...,174,menurut pengalaman saya yang pernah mengajar d...
2,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@AhWeBe,"Dulu ada istilah, Indonesia tidak kkurangan or...",129,dulu ada istilah indonesia tidak kkurangan orn...
3,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@LonelyMountains346,Jadi keinget ada teori \nbahwa sengaja pendidi...,114,jadi keinget ada teori bahwa sengaja pendidika...
4,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@ditaadityaputri7476,Wah keren sekali ka idenya. Sebagai guru saya ...,1327,wah keren sekali ka idenya sebagai guru saya s...
5,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@ArraSoso,"Setelah scroll komentar, sangat mencerminkan ...",254,setelah scroll komentar sangat mencerminkan ap...
6,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@D88FG,"Selain banyak faktor, faktor utama yaitu guru ...",628,selain banyak faktor faktor utama yaitu guru t...
7,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@addsmann,"3:39 Pernah ikut seminari oleh PPI Oslo, Finla...",572,pernah ikut seminari oleh ppi oslo finlandia i...
8,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@ramaiero7398,"Sbg seorang guru, udah mati2an ngasih materi j...",475,sbg seorang guru udah mati an ngasih materi ju...
9,Fondasi Keropos Pendidikan Indonesia,https://www.youtube.com/watch?v=3g2YoTBrQwI,@dada7064,"Pejabat : ""kami tidak akan membiarkan rakyat p...",157,pejabat kami tidak akan membiarkan rakyat pint...


In [1]:
!git --version

git version 2.55.0.windows.5


In [2]:
import os

print("Lokasi folder:", os.getcwd())
print("Daftar file:", os.listdir())

Lokasi folder: C:\Users\LENOVO
Daftar file: ['.anaconda', '.arduinoIDE', '.cache', '.claude', '.claude.json', '.conda', '.continuum', '.copilot', '.designer', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.m2', '.templateengine', '.VirtualBox', '.vscode', '.vscode-shared', '3D Objects', 'anaconda3', 'anaconda_projects', 'AppData', 'Application Data', 'Contacts', 'Cookies', 'Data Cleaning.ipynb', 'data sains.ipynb', 'dataset_cania_2979_clean.csv', 'Desktop', 'Documents', 'Downloads', 'EDA in Pandas.ipynb', 'Favorites', 'IntelGraphicsProfiles', 'Links', 'Local Settings', 'Microsoft', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TM.blf', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'Pictures', 'PrintHood', 'Recen